In [ ]:
# =====================================================
# 1. 라이브러리
# =====================================================

import re
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

In [ ]:
# =====================================================
# 2. 데이터 불러오기
# =====================================================

INPUT_FILE = "arsenal_youtube_cafe_final_labeled.xlsx"
OUTPUT_FILE = "arsenal_cafe_bertopic_result.xlsx"

df = pd.read_excel(INPUT_FILE)

print("전체 데이터:", df.shape)
print(df.columns.tolist())

# 카페 데이터만 사용
cafe_df = df[df["source"] == "cafe"].copy()

print("카페 데이터:", cafe_df.shape)
print(cafe_df["period"].value_counts(dropna=False))
print(cafe_df["gpt_sentiment"].value_counts(dropna=False))

display(cafe_df.head())

In [ ]:
# =====================================================
# 3. 카페 데이터 기본 정리
# =====================================================

topic_df = cafe_df.copy()

# 감정 라벨 정리
topic_df["gpt_sentiment"] = (
    topic_df["gpt_sentiment"]
    .fillna("neutral")
    .astype(str)
    .str.lower()
    .str.strip()
    .replace({
        "긍정": "positive",
        "중립": "neutral",
        "부정": "negative"
    })
)

# period 정리
topic_df["topic_period"] = topic_df["period"].fillna("unknown").astype(str)

# 텍스트 정리
topic_df["text_clean"] = topic_df["text_clean"].fillna("").astype(str)

# 너무 짧은 텍스트 제거
topic_df = topic_df[topic_df["text_clean"].str.len() >= 5].reset_index(drop=True)

print("토픽모델링 대상 카페 데이터:", topic_df.shape)
print(topic_df["topic_period"].value_counts())
print(topic_df["gpt_sentiment"].value_counts())

In [ ]:
# =====================================================
# 4. 한국어 토큰화 설정
# =====================================================

try:
    from kiwipiepy import Kiwi
    kiwi = Kiwi()
    USE_KIWI = True
    print("✅ Kiwi 사용")
except:
    USE_KIWI = False
    print("⚠️ Kiwi 없음: 기본 정규식 토큰화 사용")

custom_words = [
    # 팀/팬덤
    "아스날", "거너스", "구너스", "아스날조롱",

    # 감독/레전드
    "아르테타", "미켈아르테타", "벵거", "아르센벵거",

    # 아스날 선수단
    "가브리엘마갈량이스", "마갈량이스",
    "가브리엘마르티넬리", "마르티넬리",
    "가브리엘제주스",
    "노니마두에케", "마두에케",
    "다비드라야", "라야",
    "데클란라이스", "라이스",
    "레안드로트로사르", "트로사르",
    "리카르도칼라피오리", "칼라피오리",
    "마르틴수비멘디", "수비멘디",
    "마르틴외데고르", "외데고르",
    "마일스루이스스켈리", "루이스스켈리",
    "막스다우만", "다우만",
    "말리샐먼", "샐먼",
    "미켈메리노", "메리노",
    "벤화이트", "화이트",
    "부카요사카", "사카",
    "빅토르요케레스", "요케레스",
    "안드레해리먼아누스", "해리먼아누스",
    "에베레치에제", "에제",
    "위리엔팀버르", "팀버",
    "윌리엄살리바", "살리바",
    "이선은와네리", "은와네리",
    "카이하베르츠", "하베르츠",
    "케파아리사발라가", "케파",
    "크리스티안뇌가르", "뇌가르",
    "크리스티안모스케라", "모스케라",
    "토미셋포드", "셋포드",
    "피에로인카피에", "인카피에",
    "야쿠브키비오르", "키비오르",

    # 외부 영입 후보 / 비교 대상
    "바르콜라", "크루피", "토날리", "모건로저스",
    "고든", "스펜스", "스톤스", "아놀드", "케인", "토니", "벨링엄"
]

for word in custom_words:
    try:
        kiwi.add_user_word(word, "NNP")
    except:
        pass


stopwords = set([
    # =====================================================
    # 1. 일반 감탄/대화체/의미 약한 표현
    # =====================================================
    "진짜", "너무", "정말", "그냥", "뭔가", "약간", "조금", "정도", "완전", "계속",
    "오늘", "지금", "다시", "이번", "이제", "이후", "당시", "처음", "결국", "동안",
    "이거", "저거", "그거", "이런", "저런", "그런", "여기", "저기", "거기", "다음",
    "사람", "자신", "본인", "모두", "부분", "전체", "이유", "의미", "느낌",
    "마음", "생각", "이야기", "말씀", "말", "하나", "모습", "수준", "이상",
    "ㅋㅋ", "ㅎㅎ", "ㅠㅠ", "ㅜㅜ", "개추", "의외", "덕분", "의문", "사실", 

    # =====================================================
    # 2. 카페/네이버/UI/관리 관련 노이즈
    # =====================================================
    "아스날리쉬", "카페", "네이버", "본문", "댓글", "게시글", "게시", "글", "작성",
    "작성자", "조회", "추천", "공지", "규정", "운영", "관리", "관리자", "진행자",
    "알림", "설정", "해제", "클린봇", "신고", "공유", "링크", "출처", "사진",
    "이미지", "영상", "페이지", "화면", "확인", "선택", "버튼", "메뉴", "목록",
    "구독", "멤버", "회원", "방문", "인기", "베스트", "best",  "프로필",
    "내용", "정보", "질문", "답변", "기록", "검색", "번역", "인쇄", "이름",
    "자격", "단계", "준비", "예정", "상태", "경우", "관련", "사용", "방식",
    "주요", "내용", "코멘트", "저작권법", "저작권", "법", "매너", "저작", "준수", 

    # =====================================================
    # 3. 너무 일반적인 축구 단어
    # =====================================================
    "축구", "경기", "시즌", "리그", "프리미어리그", "프리미어", "라운드",
    "전반", "후반", "경기장", "라인업", "선발", "벤치", "주전", "후보",
    "선수", "감독", "팀", "구단", "팬", "응원",
    "골", "득점", "실점", "승리", "패배", "무승부", "결과",
    "수비", "공격", "전술", "교체", "출전", "부상", "복귀",

    # =====================================================
    # Kiwi가 선수/팀명을 잘못 쪼갠 토큰 & 이상용어 추가 제거
    # =====================================================
    "아스", "테타", "세트", "피스", "도련님", "아르",

    # =====================================================
    # 4. 타 팀명/대회명/지역명 노이즈
    # =====================================================
    "토트넘", "스퍼스", "맨유", "맨시티", "리버풀", "첼시", "뉴캐슬", "시티", "클롭"
    "브라이튼", "빌라", "에버튼", "풀럼", "본머스", "웨스트햄",
    "레알", "바르샤", "바르셀로나", "뮌헨", "psg",
    "잉글랜드", "런던", "프랑스", "스페인", "독일",

])

stopwords = set(str(w).strip().lower() for w in stopwords)


def basic_clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def is_stopword(word):
    word = str(word).strip().lower()

    if word in stopwords:
        return True

    # 복합어 제거: 예) 토트넘팬, 게시글작성
    for sw in stopwords:
        if len(sw) >= 2 and sw in word:
            return True

    return False


def kiwi_tokenizer(text):
    text = basic_clean_text(text)
    tokens = []

    if USE_KIWI:
        for token in kiwi.tokenize(text):
            # NNG 일반명사, NNP 고유명사, SL 외국어
            if token.tag in ["NNG", "NNP", "SL"]:
                word = token.form.strip().lower()

                if len(word) >= 2 and not is_stopword(word):
                    tokens.append(word)

    else:
        raw_tokens = re.findall(r"[가-힣a-zA-Z]{2,}", text)
        tokens = [
            t.strip().lower()
            for t in raw_tokens
            if len(t.strip()) >= 2 and not is_stopword(t)
        ]

    return tokens

In [ ]:
# =====================================================
# 카페 게시글 상단 공지문 제거 함수
# =====================================================

NOTICE_PATTERNS = [
    # 악성 댓글 / 알림 관련 문구
    r"이\s*악성\s*댓글을\s*감지합니다\.?",
    r"새로운\s*댓글이\s*추가될\s*때\s*알림을\s*보내드립니다\.?",
    r"댓글\s*알림\s*설정이\s*해제되었습니다\.?",

    # 위 문구가 한 줄로 붙어 있는 경우
    r"이\s*악성\s*댓글을\s*감지합니다\.?\s*새로운\s*댓글이\s*추가될\s*때\s*알림을\s*보내드립니다\.?\s*댓글\s*알림\s*설정이\s*해제되었습니다\.?",

    # 출처 / 주요 내용 템플릿
    r"⊙\s*출처\s*▶\s*⊙\s*주요\s*내용\s*▶?",
    r"⊙\s*출처\s*▶?",
    r"⊙\s*출처?",
    r"⊙\s*출처\s*▶?",
    
    # 특수문자가 이미 지워진 text_clean 대응
    r"출처\s*주요\s*내용",
    r"출처\s*주요\s*내용\s*▶?",

]


def remove_cafe_notice(text):
    text = str(text)

    for pattern in NOTICE_PATTERNS:
        text = re.sub(pattern, " ", text)

    # 혹시 공지문이 한 줄로 붙어 있는 경우 추가 제거
    text = re.sub(
        r"짧은\s*글.*?게시글\s*작성",
        " ",
        text
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
# =====================================================
# 카페 공지문 제거 적용
# =====================================================

topic_df["topic_text"] = topic_df["text_clean"].fillna("").astype(str)

# cafe_post에만 공지문 제거
post_mask = topic_df["content_type"].eq("cafe_post")

topic_df.loc[post_mask, "topic_text"] = topic_df.loc[
    post_mask,
    "topic_text"
].apply(remove_cafe_notice)

# 너무 짧아진 텍스트 제거
topic_df = topic_df[
    topic_df["topic_text"].str.len() >= 5
].reset_index(drop=True)

print("공지문 제거 후 토픽모델링 대상:", topic_df.shape)

display(
    topic_df.loc[
        topic_df["content_type"].eq("cafe_post"),
        ["title", "text_clean", "topic_text"]
    ].head()
)

In [ ]:
custom_replace_dict = {
    # 팀명/팬덤
    "아스널": "아스날",
    "구너": "구너스",
    "개스날": "아스날조롱",
    "무관스날": "아스날조롱",
    "개집": "아스날조롱",

    # 감독/레전드/별명
    "미켈": "아르테타",
    "테타": "아르테타",
    "벵버지": "벵거",
    "아버지": "벵거",

    # 선수명 표기 흔들림
    "외데": "외데고르",
    "와데고르": "외데고르",
    "데클란": "라이스",
    "데클런": "라이스",
    "마르티": "마르티넬리",
    "말티": "마르티넬리",
    "마르티 넬리": "마르티넬리",
    "제수스": "가브리엘제주스",
    "제주스": "가브리엘제주스",
    "칼피": "칼라피오리",
    "칼라피": "칼라피오리",
    "칼라피 오리": "칼라피오리",
    "루이스 스켈리": "루이스스켈리",
    "스켈리": "루이스스켈리",
    "팀버르": "팀버",
    "수멘": "수비멘디",
    "마두 에케": "마두에케",
    "요케 레스": "요케레스",
    "에베레치에제": "에제",
    "eze": "에제",
    "요케": "요케레스",
    "트싸": "트로사르",
    "화이트": "벤화이트",
    "벤 화이트": "벤화이트",

    # 외부 영입 후보 표기
    "모건 로저스": "모건로저스",
    "로저스": "모건로저스",
    "모건": "모건로저스",
    "바르 콜라": "바르콜라",
    "크루 피": "크루피"
}

def normalize_football_terms(text):
    text = str(text)

    for src, tgt in custom_replace_dict.items():
        text = text.replace(src, tgt)

    return text

topic_df["topic_text"] = topic_df["topic_text"].apply(normalize_football_terms)

In [ ]:
# =====================================================
# 5. BERTopic 입력 문서 준비
# =====================================================

docs = topic_df["topic_text"].tolist()

print("문서 수:", len(docs))
print(docs[:3])

In [ ]:
# =====================================================
# 6. BERTopic 모델 설정
# =====================================================

# 한국어 문장 임베딩 모델
embedding_model = SentenceTransformer("jhgan/ko-sroberta-multitask")

# 문서 수에 따라 min_df 자동 조정
min_df_value = 2 if len(docs) < 3000 else 3

vectorizer_model = CountVectorizer(
    tokenizer=kiwi_tokenizer,
    token_pattern=None,
    min_df=min_df_value,
    max_df=0.85,
    max_features=7000
)

umap_model = UMAP(
    n_neighbors=25,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=25,
    min_samples=3,
    metric="euclidean",
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)

In [ ]:
# =====================================================
# 7. BERTopic 학습 실행
# =====================================================

topics, probs = topic_model.fit_transform(docs)

topic_df["topic"] = topics

print("토픽 부여 완료")
print(topic_df["topic"].value_counts().head(20))

In [ ]:
# =====================================================
# 8. Topic -1 줄이기
# =====================================================

print("기존 -1 개수:", (topic_df["topic"] == -1).sum())

try:
    new_topics = topic_model.reduce_outliers(
        docs,
        topics,
        strategy="c-tf-idf"
    )

    topic_model.update_topics(
        docs,
        topics=new_topics,
        vectorizer_model=vectorizer_model
    )

    topic_df["topic"] = new_topics

    print("수정 후 -1 개수:", (topic_df["topic"] == -1).sum())

except Exception as e:
    print("outlier reduction 실패:", e)

In [ ]:
# =====================================================
# 9. 토픽 정보 확인
# =====================================================

topic_info = topic_model.get_topic_info()

display(topic_info.head(30))

In [ ]:
# =====================================================
# 10. 토픽별 키워드 확인
# =====================================================

for topic_id in topic_info["Topic"].head(20):
    if topic_id == -1:
        continue

    print(f"\n===== Topic {topic_id} =====")
    print(topic_model.get_topic(topic_id))

In [ ]:
# =====================================================
# 11. 토픽별 대표 문장 확인
# =====================================================

for topic_id in topic_info["Topic"].head(10):
    if topic_id == -1:
        continue

    print(f"\n\n===== Topic {topic_id} 대표 문장 =====")

    rep_docs = topic_model.get_representative_docs(topic_id)

    for doc in rep_docs[:5]:
        print("-", doc[:200])

In [ ]:
# =====================================================
# 12. 토픽명 생성
# =====================================================

def make_topic_name(topic_id):
    if topic_id == -1:
        return "기타/분류불가"

    words = topic_model.get_topic(topic_id)

    if not words:
        return f"Topic {topic_id}"

    top_words = [w for w, score in words[:4]]

    return f"Topic {topic_id}: " + " / ".join(top_words)


topic_name_map = {
    topic_id: make_topic_name(topic_id)
    for topic_id in topic_df["topic"].unique()
}

topic_df["topic_name"] = topic_df["topic"].map(topic_name_map)

display(topic_df[["topic_period", "gpt_sentiment", "text_clean", "topic", "topic_name"]].head())

In [ ]:
# =====================================================
# 13. 38R 전후 토픽 비교
# =====================================================

valid_topic_df = topic_df[topic_df["topic"] != -1].copy()

period_topic_counts = pd.crosstab(
    valid_topic_df["topic_period"],
    valid_topic_df["topic_name"]
)

period_topic_ratio = pd.crosstab(
    valid_topic_df["topic_period"],
    valid_topic_df["topic_name"],
    normalize="index"
) * 100

display(period_topic_counts)
display(period_topic_ratio.round(2))

In [ ]:
# =====================================================
# 14. 38R 전후 Top 토픽 테이블
# =====================================================

def get_top_topics_by_group(data, group_col, top_n=10):
    result = []

    for group, group_df in data.groupby(group_col):
        counts = group_df["topic_name"].value_counts().head(top_n)
        total = len(group_df)

        for topic_name, count in counts.items():
            result.append({
                group_col: group,
                "topic_name": topic_name,
                "count": count,
                "ratio": round(count / total * 100, 2)
            })

    return pd.DataFrame(result)


period_top_topics = get_top_topics_by_group(
    valid_topic_df,
    "topic_period",
    top_n=10
)

display(period_top_topics)

In [ ]:
# =====================================================
# 15. 긍정 / 부정별 토픽 비교
# =====================================================

pn_topic_df = valid_topic_df[
    valid_topic_df["gpt_sentiment"].isin(["positive", "negative"])
].copy()

sentiment_topic_counts = pd.crosstab(
    pn_topic_df["gpt_sentiment"],
    pn_topic_df["topic_name"]
)

sentiment_topic_ratio = pd.crosstab(
    pn_topic_df["gpt_sentiment"],
    pn_topic_df["topic_name"],
    normalize="index"
) * 100

display(sentiment_topic_counts)
display(sentiment_topic_ratio.round(2))

In [ ]:
# =====================================================
# 16. 긍정/부정별 Top 토픽 테이블
# =====================================================

sentiment_top_topics = get_top_topics_by_group(
    pn_topic_df,
    "gpt_sentiment",
    top_n=10
)

display(sentiment_top_topics)

In [ ]:
# =====================================================
# 17. 한글 폰트 설정
# =====================================================

def set_korean_font():
    font_candidates = [
        "Malgun Gothic",
        "AppleGothic",
        "NanumGothic",
        "Noto Sans CJK KR"
    ]

    installed_fonts = [f.name for f in fm.fontManager.ttflist]

    for font in font_candidates:
        if font in installed_fonts:
            plt.rcParams["font.family"] = font
            plt.rcParams["axes.unicode_minus"] = False
            print(f"✅ 한글 폰트 설정 완료: {font}")
            return

    print("⚠️ 한글 폰트를 찾지 못했습니다.")

set_korean_font()

In [ ]:
# =====================================================
# 18. 카페 전체 Top 토픽 막대그래프
# =====================================================

top_topic_counts = (
    valid_topic_df["topic_name"]
    .value_counts()
    .head(12)
    .sort_values()
)

plt.figure(figsize=(10, 7))
plt.barh(top_topic_counts.index, top_topic_counts.values)
plt.title("팬 카페 주요 토픽 Top 12", fontsize=16, fontweight="bold")
plt.xlabel("문서 수")
plt.ylabel("토픽")
plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================
# 19. 38R 전후 주요 토픽 비율 비교
# =====================================================

top_topics = (
    valid_topic_df["topic_name"]
    .value_counts()
    .head(8)
    .index
)

period_compare = pd.crosstab(
    valid_topic_df["topic_period"],
    valid_topic_df["topic_name"],
    normalize="index"
) * 100

period_compare = period_compare.reindex(index=["before_38R", "after_38R"])
period_compare = period_compare[top_topics].fillna(0)

display(period_compare.round(2))

period_compare.T.plot(
    kind="barh",
    figsize=(10, 7)
)

plt.title("팬 카페 38R 전후 주요 토픽 비율 비교", fontsize=16, fontweight="bold")
plt.xlabel("비율(%)")
plt.ylabel("토픽")
plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================
# 20. 긍정/부정 주요 토픽 비율 비교
# =====================================================

top_topics_pn = (
    pn_topic_df["topic_name"]
    .value_counts()
    .head(8)
    .index
)

sentiment_compare = pd.crosstab(
    pn_topic_df["gpt_sentiment"],
    pn_topic_df["topic_name"],
    normalize="index"
) * 100

sentiment_compare = sentiment_compare.reindex(index=["positive", "negative"])
sentiment_compare = sentiment_compare[top_topics_pn].fillna(0)

display(sentiment_compare.round(2))

sentiment_compare.T.plot(
    kind="barh",
    figsize=(10, 7)
)

plt.title("팬 카페 긍정/부정 감정별 주요 토픽 비율 비교", fontsize=16, fontweight="bold")
plt.xlabel("비율(%)")
plt.ylabel("토픽")
plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================
# 21. 결과 저장
# =====================================================

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    topic_df.to_excel(writer, sheet_name="cafe_topic_labeled_data", index=False)
    topic_info.to_excel(writer, sheet_name="topic_info", index=False)
    period_top_topics.to_excel(writer, sheet_name="period_top_topics", index=False)
    sentiment_top_topics.to_excel(writer, sheet_name="sentiment_top_topics", index=False)
    period_topic_ratio.round(2).to_excel(writer, sheet_name="period_topic_ratio")
    sentiment_topic_ratio.round(2).to_excel(writer, sheet_name="sentiment_topic_ratio")

print("✅ 저장 완료:", OUTPUT_FILE)

In [ ]:
display(topic_info.head(20))
display(period_top_topics)
display(sentiment_top_topics)